In [11]:
from typing import List
import glob
import os
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [12]:
NUMBER_OF_FLOWS = 50
STEPS = 5
INTERVAL_LENGTH = int(NUMBER_OF_FLOWS / STEPS)
SESSIONS = np.arange(0, NUMBER_OF_FLOWS, step=INTERVAL_LENGTH)

In [13]:
def acceptance_rate(ar_array) -> List:
    slices = np.array_split(ar_array, INTERVAL_LENGTH)
    ar_result = []
    for slice in slices:
        values, counts = np.unique(slice, return_counts=True)
        if len(counts) == 1:
            if values[0] == 0:
                acc_rate = 0
            elif values[0] == 1:
                acc_rate = 1
        else:
            count_succ = counts[1]
            count_fail = counts[0]
            acc_rate = count_succ / (count_succ + count_fail)
        ar_result.append(acc_rate)
    return ar_result

In [52]:
def resource_utilization(ar_array) -> List:
    slices = np.array_split(ar_array, INTERVAL_LENGTH)
    resource_result = []
    for slice in slices:
        if len(slice) == 0:
            continue
        average_consumption = np.mean(slice)
        resource_result.append(average_consumption)
    return resource_result

# One speed analysis

In [15]:
results_flows_directories = glob.glob("../results_flows_speed_60.0_*/*")
results_flows_directories

['..\\results_flows_speed_60.0_50\\dp_36_sfc_on_speed_60.0_con_1']

In [16]:
def dataframe_dados(direc):
    df = pd.DataFrame()
    for alg_dir in direc:
        files = os.listdir(alg_dir)
        simu_exec_name = alg_dir.split("/")[-1].split("_")
        alg_name = simu_exec_name[0]
        sfc = simu_exec_name[3]
        con = simu_exec_name[-1]
        speed = simu_exec_name[-3]

        for idx, file in enumerate(files):
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
            except:
                continue

            simulation_df["algorithm"] = alg_name
            simulation_df["sfc"] = sfc
            simulation_df["speed"] = speed
            simulation_df["exec"] = idx
            simulation_df["connections"] = con

            df = pd.concat([df, simulation_df], axis=0)

    df["dep"] = "deploy"
    df.loc[df["sfc_id"].duplicated(keep="first"), "dep"] = "redeploy"

    grouped_df = df.groupby(["sfc", "algorithm", "speed", "dep", "connections"])

    return grouped_df

In [17]:
grouped_flows = [dataframe_dados([x]) for x in results_flows_directories]

In [18]:
def format_grouped_df(grouped_df):
    acceptance_series = grouped_df["success"].apply(resource_utilization)
    bd_series = grouped_df["bandwidth_utilization"].apply(resource_utilization)
    cache_series = grouped_df["cache_utilization"].apply(resource_utilization)
    cpu_series = grouped_df["cpu_utilization"].apply(resource_utilization)
    dic = {
        "acceptance": acceptance_series,
        "bd": bd_series,
        "cpu": cpu_series,
        "cache": cache_series,
    }
    return dic

In [19]:
formated_flows = [format_grouped_df(x) for x in grouped_flows]

TypeError: unsupported operand type(s) for +: 'int' and 'str'

## Line Graph

In [ ]:
acceptance = [x["acceptance"] for x in formated_flows]
cpu = [x["cpu"] for x in formated_flows]
bd = [x["bd"] for x in formated_flows]
cache = [x["cache"] for x in formated_flows]

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_resource_utilization_simulations(
    resource_series_list, plot_type, title, xlim=None, ylim=None, figsize=(1000, 600)
):
    if plot_type == "sep":
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Deploy", "Redeploy"))
    else:
        fig = go.Figure()

    dic = {}
    dic_else = {}
    colors = {
        "0": "blue",
        "2": "green",
        "4": "orange",
        "6": "red",
    }  # Definição das cores para cada 'con'
    x = list(range(5, 55, 5))

    for resource_series in resource_series_list:
        if plot_type == "sep":
            for dep in resource_series.index.get_level_values("dep"):
                con = resource_series.index.get_level_values("connections")[0]
                ar = pd.DataFrame(resource_series)
                ar = ar.query("('{0}' == dep) & (connections == '{1}')".format(dep, con))
                ar = ar.values.tolist()
                partitions = np.array_split(ar, len(ar))
                stacked_partitions = np.vstack(partitions)
                result = (np.sum(stacked_partitions, axis=0) / len(stacked_partitions)).ravel()
                dic[(con, dep)] = result
        else:
            con = resource_series.index.get_level_values("connections")[0]
            ar = resource_series.values.tolist()
            partitions = np.array_split(ar, len(ar))
            stacked_partitions = np.vstack(partitions)
            result = (np.sum(stacked_partitions, axis=0) / len(stacked_partitions)).ravel()
            dic_else[con] = result

    if plot_type == "sep":
        order_dic = dict(sorted(dic.items(), key=lambda x: x[0][0]))
        for key, data in order_dic.items():
            con = str(key[0])
            name = "connections: " + con
            line_color = colors[con]
            if key[1] == "deploy":
                fig.add_trace(
                    go.Scatter(x=x, y=data, mode="lines", name=name, line=dict(color=line_color)),
                    row=1,
                    col=1,
                )
            elif key[1] == "redeploy":
                fig.add_trace(
                    go.Scatter(x=x, y=data, mode="lines", name=name, line=dict(color=line_color)),
                    row=1,
                    col=2,
                )
        fig.update_traces(showlegend=False, row=1, col=2)  # Oculta a legenda do subplot "redeploy"
    else:
        order_dic = dict(sorted(dic_else.items(), key=lambda x: x[0]))
        for key, data in order_dic.items():
            name = "connections: " + str(key)

            fig.add_trace(
                go.Scatter(x=x, y=data, mode="lines", name=name, line=dict(color=line_color))
            )

    fig.update_layout(
        height=figsize[1],
        width=figsize[0],
        title=title,
        xaxis_title="Sessions",
        font=dict(size=16),
        title_font=dict(size=20),
        legend_font=dict(size=16),
    )

    if xlim is not None:
        fig.update_xaxes(range=xlim)
    if ylim is not None:
        fig.update_yaxes(range=ylim)

    fig.show()

In [ ]:
# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
plot_resource_utilization_simulations(
    acceptance, title="Acceptance", plot_type="sep", ylim=[0.6, 1], figsize=(1000, 400)
)

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
plot_resource_utilization_simulations(
    cpu, title="CPU", plot_type="sep", ylim=[0.15, 0.4], figsize=(1000, 400)
)

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
plot_resource_utilization_simulations(
    bd, title="Bandwidth", plot_type="sep", ylim=[0.02, 0.15], figsize=(1000, 400)
)

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
plot_resource_utilization_simulations(
    cache, title="Cache", plot_type="sep", ylim=[0.05, 0.25], figsize=(1000, 400)
)

## Boxplot

In [ ]:
colors = {"0": "blue", "2": "green", "4": "orange", "6": "red"}


def create_boxplot(
    resource_series_list, title, plot_type, figsize=(1000, 600), xlim=None, ylim=None
):
    if plot_type == "sep":
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Deploy", "Redeploy"))
    else:
        fig = go.Figure()

    dic = {}
    dic_else = {}

    for resource_series in resource_series_list:
        if plot_type == "sep":
            for dep in resource_series.index.get_level_values("dep"):
                con = resource_series.index.get_level_values("connections")[0]
                ar = pd.DataFrame(resource_series)
                ar = ar.query("('{0}' == dep) & (connections == '{1}')".format(dep, con))
                ar = ar.values.tolist()
                partitions = np.array_split(ar, len(ar))
                stacked_partitions = np.vstack(partitions)
                result = (np.sum(stacked_partitions, axis=0) / len(stacked_partitions)).ravel()
                dic[(con, dep)] = result
        else:
            con = resource_series.index.get_level_values("connections")[0]
            ar = resource_series.values.tolist()
            partitions = np.array_split(ar, len(ar))
            stacked_partitions = np.vstack(partitions)
            result = (np.sum(stacked_partitions, axis=0) / len(stacked_partitions)).ravel()
            dic_else[con] = result

    if plot_type == "sep":
        order_dic = dict(sorted(dic.items(), key=lambda x: x[0][0]))
        for key, data in order_dic.items():
            con = str(key[0])
            x_label = con  # Adiciona o número de connections ao rótulo
            if key[1] == "deploy":
                fig.add_trace(
                    go.Box(y=data, x=[x_label] * len(data), marker=dict(color=colors[con])),
                    row=1,
                    col=1,
                )
            elif key[1] == "redeploy":
                fig.add_trace(
                    go.Box(y=data, x=[x_label] * len(data), marker=dict(color=colors[con])),
                    row=1,
                    col=2,
                )
    else:
        order_dic = dict(sorted(dic_else.items(), key=lambda x: x[0]))
        for key, data in order_dic.items():
            x_label = key  # Adiciona o número de connections ao rótulo
            fig.add_trace(
                go.Box(y=data, x=[x_label] * len(data), name=key, marker=dict(color=colors[key]))
            )

    fig.update_layout(
        height=figsize[1],
        width=figsize[0],
        title=title,
        xaxis=dict(title="Connections"),
        font=dict(size=16),
        title_font=dict(size=20),
        legend_font=dict(size=16),
        showlegend=False,
    )

    if xlim is not None:
        fig.update_xaxes(range=xlim)
    if ylim is not None:
        fig.update_yaxes(range=ylim)

    fig.show()

In [ ]:
# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
create_boxplot(acceptance, title="Acceptance", plot_type="sep", ylim=[0.6, 1], figsize=(1000, 400))

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
create_boxplot(cpu, title="CPU", plot_type="sep", ylim=[0.15, 0.4], figsize=(1000, 400))

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
create_boxplot(bd, title="Bandwidth", plot_type="sep", ylim=[0.02, 0.15], figsize=(1000, 400))

# Chamada para o gráfico de acceptance com xlim, ylim e figsize definidos
create_boxplot(cache, title="Cache", plot_type="sep", ylim=[0.05, 0.25], figsize=(1000, 400))

# More Speed

In [53]:
from typing import List
import glob
import numpy as np
import warnings
import plotly.express as px

warnings.filterwarnings("ignore")

In [54]:
results_flows_directories = glob.glob("..//results_flows_speed_*//*")
results_flows_directories

['..\\results_flows_speed_0.0_50\\dp_36_sfc_on_speed_0.0_con_1',
 '..\\results_flows_speed_15.0_50\\dp_36_sfc_on_speed_15.0_con_1',
 '..\\results_flows_speed_30.0_50\\dp_36_sfc_on_speed_30.0_con_1',
 '..\\results_flows_speed_45.0_50\\dp_36_sfc_on_speed_45.0_con_1',
 '..\\results_flows_speed_60.0_50\\dp_36_sfc_on_speed_60.0_con_1',
 '..\\results_flows_speed_75.0_50\\dp_36_sfc_on_speed_75.0_con_1']

In [55]:
df = pd.DataFrame()

for alg_dir in results_flows_directories:
    files = os.listdir(alg_dir)
    simu_exec_name = alg_dir.split("\\")[-1].split("_")

    alg_name = simu_exec_name[0]
    sfc = simu_exec_name[3]
    vel = simu_exec_name[-3]
    con = simu_exec_name[-1]
    print(sfc, alg_name, vel, con)
    for file in files:
        data_path = os.path.join(alg_dir, file)
        try:
            simulation_df = pd.read_csv(data_path)
        except:
            continue

        simulation_df["algorithm"] = alg_name
        simulation_df["sfc"] = sfc
        simulation_df["speed"] = vel
        simulation_df["connections"] = con
        simulation_df["dep"] = "deploy"
        simulation_df.loc[simulation_df["sfc_id"].duplicated(keep="first"), "dep"] = "redeploy"
        print()
        df = pd.concat([df, simulation_df], axis=0)

on dp 0.0 1


















































on dp 15.0 1

















































on dp 30.0 1


















































on dp 45.0 1


















































on dp 60.0 1

















































on dp 75.0 1




















































In [56]:
grouped_df = df.groupby(["sfc", "algorithm", "speed", "dep", "connections"])

In [57]:
def resource_utilization(ar_array) -> List:
    slices = np.array_split(ar_array, INTERVAL_LENGTH)
    resource_result = []
    for slice in slices:
        if len(slice) == 0:
            continue
        average_consumption = np.mean(slice)
        resource_result.append(average_consumption)
    return resource_result


acceptance_series = pd.DataFrame(grouped_df["success"].apply(resource_utilization))
bd_series = pd.DataFrame(grouped_df["bandwidth_utilization"].apply(resource_utilization))
cache_series = pd.DataFrame(grouped_df["cache_utilization"].apply(resource_utilization))
cpu_series = pd.DataFrame(grouped_df["cpu_utilization"].apply(resource_utilization))
decision_time_series = pd.DataFrame(grouped_df["duration"].apply(resource_utilization))

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
seps_types = ["deploy", "redeploy"]

acc_deploy = [acceptance_series.query(f"(dep=='{sep_type}')") for sep_type in seps_types]
bd_deploy = [bd_series.query(f"(dep=='{sep_type}')") for sep_type in seps_types]
cache_deploy = [cache_series.query(f"(dep=='{sep_type}')") for sep_type in seps_types]
cpu_deploy = [cpu_series.query(f"(dep=='{sep_type}')") for sep_type in seps_types]
decision_deploy = [decision_time_series.query(f"(dep=='{sep_type}')") for sep_type in seps_types]

In [ ]:
def format_boxplot(df_list, value):
    lista_resultado = []
    for my_df in df_list:
        speeds = sorted([x for x in np.unique(my_df.index.get_level_values("speed"))])
        conections = sorted([x for x in np.unique(my_df.index.get_level_values("connections"))])
        dic = pd.DataFrame()

        for con in conections:
            d_velocitys = pd.DataFrame()
            for sp in speeds:
                ar = my_df.query("('{0}' == speed) & (connections == '{1}')".format(sp, con))
                ar = ar.values.tolist()
                partitions = np.array_split(np.array(ar), len(ar))
                stacked_partitions = np.vstack(partitions)
                result = (np.sum(stacked_partitions, axis=0) / len(stacked_partitions)).ravel()
                d_velocitys[sp] = result
            d_velocitys["Connections"] = con
            dic = pd.concat([dic, d_velocitys])

        df_melted = dic.melt(id_vars="Connections", var_name="Speed (km/h)", value_name=value)
        lista_resultado.append(df_melted)
    return lista_resultado

In [ ]:
acc_dep = format_boxplot(acc_deploy, "Acceptance")
cpu_dep = format_boxplot(cpu_deploy, "CPU")
bd_dep = format_boxplot(bd_deploy, "Bandwidth")
cache_dep = format_boxplot(cache_deploy, "Cache")
decision_dep = format_boxplot(decision_deploy, "Decision Time")

dep_dict = {
    "acc": acc_dep,
    "cpu": cpu_dep,
    "bd": bd_dep,
    "cache": cache_dep,
    "decision": decision_dep,
}

In [ ]:
acc = format_boxplot([acceptance_series], "Acceptance")
cpu = format_boxplot([cpu_series], "CPU")
bd = format_boxplot([bd_series], "Bandwidth")
cache = format_boxplot([cache_series], "Cache")
decision = format_boxplot([decision_time_series], "Decision Time")

dict = {"acc": acc, "cpu": cpu, "bd": bd, "cache": cache, "decision": decision}

In [ ]:


def all_boxplot(plots_dict, speeds_selected=[]):
    def speeds_boxplot(my_data, metric, speeds=[]):
        num_plots = len(my_data)
        fig = make_subplots(
            rows=1, cols=num_plots, subplot_titles=["deploy", "redeploy"][:num_plots]
        )

        if num_plots > 1:
            for i, plot_data in enumerate(my_data):
                df_filtered = plot_data[plot_data["Speed (km/h)"].isin(speeds)]
                fig = px.box(df_filtered, x="Connections", y=metric, color="Speed (km/h)")

        else:
            df_filtered = my_data[0][my_data[0]["Speed (km/h)"].isin(speeds)]
            fig.add_trace(
                px.box(df_filtered, x="Connections", y=metric, color="Speed (km/h)").data[0],
                row=1,
                col=1,
            )

        fig.update_traces(quartilemethod="exclusive")  # or "inclusive", or "linear" by default
        fig.update_layout(height=600, width=1000)

        fig.show()
        fig.write_image(
            f"{metric}_boxplot.png"
        )  # Nome do arquivo será o nome da métrica seguido de "_boxplot.png"

    speeds_boxplot(plots_dict["acc"], "Acceptance", speeds_selected)
    speeds_boxplot(plots_dict["cpu"], "CPU", speeds_selected)
    speeds_boxplot(plots_dict["bd"], "Bandwidth", speeds_selected)
    speeds_boxplot(plots_dict["cache"], "Cache", speeds_selected)
    speeds_boxplot(plots_dict["decision"], "Decision Time", speeds_selected)


all_boxplot(plots_dict=dep_dict, speeds_selected=["30.0", "60.0"])

In [ ]:
all_boxplot(plots_dict=dep_dict, speeds_selected=["30.0", "60.0"])

# Deploy and Redeploy